In [1]:
# BRONZE NOTEBOOK — Full Auto Discovery + Ingestion + Validation + Incremental Merge
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType
from delta.tables import DeltaTable
from datetime import datetime
import uuid

spark.conf.set("spark.sql.shuffle.partitions", "2")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# ── ONLY EDIT THESE 3 LINES ───────────────────────────────────────────────────
BASE_PATH = "Files/adventure-poc-febric/poc_adventure"      # path inside Lake_House
BRONZE_DB  = "LH_Bronze_layer.dbo"     # matches your Bronze lakehouse name
AUDIT_DB   = "LH_Gold_layer.audit"     # matches your audit schema in screenshot
# ─────────────────────────────────────────────────────────────────────────────

run_id        = str(uuid.uuid4())
pipeline_name = "Bronze_Pipeline"
master_start  = datetime.now()

print(f"[BRONZE] Started   : {master_start}")
print(f"[BRONZE] Run ID    : {run_id}")
print(f"[BRONZE] Base Path : {BASE_PATH}")
print(f"[BRONZE] Bronze DB : {BRONZE_DB}")
print(f"[BRONZE] Audit DB  : {AUDIT_DB}")

# ── Audit Schemas ─────────────────────────────────────────────────────────────
AUDIT_SCHEMA = StructType([
    StructField("ROW_ID",        StringType(),    True),
    StructField("RUN_ID",        StringType(),    True),
    StructField("CREATED_DATE",  TimestampType(), True),
    StructField("PIPELINE_NAME", StringType(),    True),
    StructField("SOURCE_TYPE",   StringType(),    True),
    StructField("SOURCE_TABLE",  StringType(),    True),
    StructField("TARGET_TABLE",  StringType(),    True),
    StructField("START_TIME",    TimestampType(), True),
    StructField("END_TIME",      TimestampType(), True),
    StructField("STATUS",        StringType(),    True),
    StructField("STATUS_DESC",   StringType(),    True),
])

COUNT_SCHEMA = StructType([
    StructField("ROW_ID",                             StringType(),    True),
    StructField("RUN_ID",                             StringType(),    True),
    StructField("SOURCE_TYPE",                        StringType(),    True),
    StructField("SOURCE_TABLE",                       StringType(),    True),
    StructField("TARGET_TABLE",                       StringType(),    True),
    StructField("LAYER",                              StringType(),    True),
    StructField("SOUREC_FILE_COUNT",                  StringType(),    True),
    StructField("STAGING_FILE_COUNT",                 StringType(),    True),
    StructField("SOURCE_TABLE_COUNT",                 LongType(),      True),
    StructField("STAGING_TABLE_COUNT",                LongType(),      True),
    StructField("ERROR_COUNT_SOURCE_TO_STAGING_FILE", LongType(),      True),
    StructField("ERROR_COUNT_SOURCE_TO_STAGING",      LongType(),      True),
    StructField("Current_time",                       TimestampType(), True),
])

# ── Helper: Table Exists ──────────────────────────────────────────────────────
def tbl_exists(t):
    try:
        spark.sql(f"DESCRIBE TABLE {t}")
        return True
    except:
        return False

# ── Helper: Audit Log ─────────────────────────────────────────────────────────
def audit(tbl, table_name, tgt, t0, s, d=""):
    try:
        row = spark.createDataFrame(
            [(str(uuid.uuid4()), run_id, datetime.now(), pipeline_name,
              BASE_PATH, table_name, tgt, t0, datetime.now(), str(s), str(d[:500]))],
            schema=AUDIT_SCHEMA
        )
        for field in AUDIT_SCHEMA.fields:
            row = row.withColumn(field.name, F.col(field.name).cast(field.dataType))
        row.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(tbl)
    except Exception as ae:
        print(f"  [AUDIT WARNING] Could not write to {tbl}: {ae}")

# ── Helper: Count Log ─────────────────────────────────────────────────────────
def count_audit(table_name, tgt, src_cnt, stg_cnt, err=0):
    try:
        row = spark.createDataFrame(
            [(str(uuid.uuid4()), run_id, BASE_PATH, table_name, tgt, "BRONZE",
              None, None, int(src_cnt), int(stg_cnt), int(err), int(err), datetime.now())],
            schema=COUNT_SCHEMA
        )
        for field in COUNT_SCHEMA.fields:
            row = row.withColumn(field.name, F.col(field.name).cast(field.dataType))
        row.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(f"{AUDIT_DB}.count_log_table")
    except Exception as ce:
        print(f"  [COUNT WARNING] Could not write count log: {ce}")

# ── Core: Process One Table ───────────────────────────────────────────────────
def process_table(table_name):

    SRC_PATH = f"{BASE_PATH}/{table_name}"
    TGT      = f"{BRONZE_DB}.{table_name}"
    t0       = datetime.now()
    status   = "SUCCESS"
    desc     = ""
    src_cnt  = 0
    stg_cnt  = 0
    tgt_cnt  = 0

    print(f"\n  [{table_name}] Reading from : {SRC_PATH}")
    print(f"  [{table_name}] Writing to   : {TGT}")

    # ── Step 1: Read CSV File ─────────────────────────────────────────────────
    try:
        raw = (spark.read
                    .option("header",      "true")
                    .option("inferSchema", "true")
                    .option("multiLine",   "true")
                    .option("escape",      '"')
                    .csv(SRC_PATH))
        src_cnt = raw.count()
        print(f"  [{table_name}] Source rows   : {src_cnt}")
        print(f"  [{table_name}] Columns       : {raw.columns}")
    except Exception as e:
        audit(f"{AUDIT_DB}.copy_audit_log",     table_name, TGT, t0, "FAILED", str(e))
        audit(f"{AUDIT_DB}.notebook_audit_log", table_name, TGT, t0, "FAILED", str(e))
        raise

    # ── Step 2: Auto-Detect PK and Load Date ─────────────────────────────────
    pk_col = next((c for c in raw.columns if c.lower().endswith("id")), raw.columns[0])
    ld_col = next((c for c in raw.columns if "date" in c.lower() or "time" in c.lower()), "")
    PKS    = [pk_col]
    print(f"  [{table_name}] PK detected   : {pk_col}")
    print(f"  [{table_name}] Load date     : {ld_col if ld_col else 'None — full load'}")

    # ── Step 3: NULL PK Check ─────────────────────────────────────────────────
    for k in PKS:
        null_count = raw.filter(F.col(k).isNull()).count()
        if null_count > 0:
            msg = f"NULL PK [{k}]: {null_count} rows found — rejecting table"
            audit(f"{AUDIT_DB}.copy_audit_log",     table_name, TGT, t0, "FAILED", msg)
            audit(f"{AUDIT_DB}.notebook_audit_log", table_name, TGT, t0, "FAILED", msg)
            raise ValueError(msg)
    print(f"  [{table_name}] NULL PK check : PASSED")

    # ── Step 4: Empty File Check ──────────────────────────────────────────────
    if src_cnt == 0:
        audit(f"{AUDIT_DB}.copy_audit_log",     table_name, TGT, t0, "NO_DATA", "Empty source file")
        count_audit(table_name, TGT, 0, 0)
        audit(f"{AUDIT_DB}.notebook_audit_log", table_name, TGT, t0, "NO_DATA", "Empty source file")
        print(f"  [{table_name}] NO_DATA — skipping")
        return "NO_DATA"

    # ── Step 5: Incremental Filter ────────────────────────────────────────────
    df = raw
    if ld_col and tbl_exists(TGT):
        try:
            max_ts = spark.sql(f"SELECT MAX(`{ld_col}`) AS m FROM {TGT}").collect()[0]["m"]
            if max_ts:
                df = df.filter(F.col(ld_col) > F.lit(max_ts))
                print(f"  [{table_name}] Incremental   : [{ld_col}] > {max_ts}")
        except Exception as ie:
            print(f"  [{table_name}] Incremental filter skipped: {ie}")

    stg_cnt = df.count()
    print(f"  [{table_name}] Rows to write : {stg_cnt}")

    # ── Step 6: Write to Bronze Delta ─────────────────────────────────────────
    # ── Step 6: Write to Bronze Delta ─────────────────────────────────────────
    try:
        if tbl_exists(TGT):
            # Incremental load → APPEND
            df.write.format("delta") \
                .mode("append") \
                .option("mergeSchema", "true") \
                .saveAsTable(TGT)

            print(f"  [{table_name}] Mode          : APPEND (incremental load)")

        else:
            # First time load → CREATE
            df.write.format("delta") \
                .mode("overwrite") \
                .saveAsTable(TGT)

            print(f"  [{table_name}] Mode          : CREATE (first load)")

        spark.sql(f"OPTIMIZE {TGT} ZORDER BY (`{PKS[0]}`)")

        tgt_cnt = spark.sql(f"SELECT COUNT(1) AS c FROM {TGT}").collect()[0]["c"]
        print(f"  [{table_name}] Target rows   : {tgt_cnt}")

    except Exception as e:
        status = "FAILED"
        desc   = str(e)
        audit(f"{AUDIT_DB}.copy_audit_log",     table_name, TGT, t0, status, desc)
        count_audit(table_name, TGT, src_cnt, stg_cnt, err=1)
        audit(f"{AUDIT_DB}.notebook_audit_log", table_name, TGT, t0, status, desc)
        raise

    # ── Step 7: Write Audit Logs ──────────────────────────────────────────────
    audit(f"{AUDIT_DB}.copy_audit_log",     table_name, TGT, t0, status, desc)
    count_audit(table_name, TGT, src_cnt, stg_cnt)
    audit(f"{AUDIT_DB}.notebook_audit_log", table_name, TGT, t0, status, desc)

    print(f"  [{table_name}] ✔ SUCCESS | src={src_cnt} stg={stg_cnt} tgt={tgt_cnt}")
    return "SUCCESS"

# ── Auto-Discover ALL Files from Lake_House/Files/poc_adventure ───────────────
# Lake_House is attached as a secondary lakehouse in this notebook
# The path "Files/poc_adventure" resolves relative to whichever lakehouse
# has that folder. If Lake_House is secondary, the path works via shortcut.
try:
    all_items = mssparkutils.fs.ls(f"Files/adventure-poc-febric/poc_adventure")
    all_tables = [
        f.name for f in all_items
        if not f.name.startswith("_") and not f.name.startswith(".")
    ]
    print(f"\n[BRONZE] Discovered via mssparkutils: {len(all_tables)} files")
except Exception as e:
    print(f"[BRONZE] mssparkutils path failed: {e}")
    # Fallback: try with /lakehouse/ prefix paths
    try:
        all_items = mssparkutils.fs.ls("/lakehouse/Lake_House/Files/poc_adventure")
        all_tables = [
            f.name for f in all_items
            if not f.name.startswith("_") and not f.name.startswith(".")
        ]
        BASE_PATH = "/lakehouse/Lake_House/Files/poc_adventure"
        print(f"[BRONZE] Discovered via full path: {len(all_tables)} files")
    except Exception as e2:
        print(f"[BRONZE] Both path attempts failed: {e2}")
        # Fallback: read from metadata table
        meta = spark.sql("""
            SELECT FILE_NAME
            FROM LH_Gold_layer.audit.source_metadata_config
            WHERE ISACTIVE = 1 AND LAYER = 'BRONZE'
            ORDER BY PRIORITY
        """)
        all_tables = [r.FILE_NAME for r in meta.collect()]
        print(f"[BRONZE] Using metadata config: {len(all_tables)} tables")

print(f"\n[BRONZE] Tables to process: {len(all_tables)}")
for t in all_tables:
    print(f"           → {t}")

# ── Run All Tables ────────────────────────────────────────────────────────────
results = []
for idx, table in enumerate(all_tables, 1):
    print(f"\n[BRONZE] ══════════════════════════════════════════════")
    print(f"[BRONZE] [{idx} / {len(all_tables)}]  {table}")
    print(f"[BRONZE] ══════════════════════════════════════════════")
    try:
        s = process_table(table)
        results.append((table, s, ""))
    except Exception as e:
        results.append((table, "FAILED", str(e)[:200]))
        print(f"  [{table}] ✘ FAILED → {e}")

# ── Final Summary ─────────────────────────────────────────────────────────────
success = sum(1 for r in results if r[1] == "SUCCESS")
failed  = sum(1 for r in results if r[1] == "FAILED")
no_data = sum(1 for r in results if r[1] == "NO_DATA")

print(f"\n[BRONZE] ══════════════════════════════════════════════════")
print(f"[BRONZE] PIPELINE COMPLETE : {datetime.now()}")
print(f"[BRONZE] Total   = {len(results)}")
print(f"[BRONZE] Success = {success}")
print(f"[BRONZE] Failed  = {failed}")
print(f"[BRONZE] No Data = {no_data}")
print(f"[BRONZE] ══════════════════════════════════════════════════")
for table, status, err in results:
    icon = "✔" if status == "SUCCESS" else ("−" if status == "NO_DATA" else "✘")
    msg  = f"→ {err}" if err else ""
    print(f"  {icon}  {table:45s}  {status}  {msg}")

if failed:
    raise Exception(f"[BRONZE] {failed} table(s) failed. See details above.")


StatementMeta(, 75ca206e-3d4f-4136-9f3e-45937bed1b80, 3, Finished, Available, Finished, False)

[BRONZE] Started   : 2026-04-20 07:43:27.897653
[BRONZE] Run ID    : acf3d5f9-e094-40b6-b6ab-7204c8e3dbc7
[BRONZE] Base Path : Files/adventure-poc-febric/poc_adventure
[BRONZE] Bronze DB : LH_Bronze_layer.dbo
[BRONZE] Audit DB  : LH_Gold_layer.audit

[BRONZE] Discovered via mssparkutils: 17 files

[BRONZE] Tables to process: 17
           → Product
           → ProductCategory
           → ProductCostHistory
           → ProductDescription
           → ProductDocument
           → ProductInventory
           → ProductListPriceHistory
           → ProductModel
           → ProductModelIllustration
           → ProductModelProductDescriptionCulture
           → ProductPhoto
           → ProductProductPhoto
           → ProductReview
           → ProductSubcategory
           → ProductVendor
           → PurchaseOrderDetail
           → PurchaseOrderHeader

[BRONZE] ══════════════════════════════════════════════
[BRONZE] [1 / 17]  Product
[BRONZE] ═════════════════════════════════════════